# PCA y validaci?n cruzada (soluci?n)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

wine = load_wine(as_frame=True)
X = wine.data.copy()
y = wine.target.copy()

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
pca = PCA()
pca.fit(X_s)

cum_var = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.searchsorted(cum_var, 0.95) + 1)
print(f"Componentes para 95% de varianza: {n95}")

plt.figure(figsize=(7.5, 4.2))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(0.95, color="r", linestyle="--")
plt.xlabel("N?mero de componentes")
plt.ylabel("Varianza acumulada")
plt.title("PCA: varianza explicada acumulada")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)

pipe_no_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, multi_class="multinomial")),
])

pipe_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=n95)),
    ("clf", LogisticRegression(max_iter=5000, multi_class="multinomial")),
])

score_no = cross_val_score(pipe_no_pca, X, y, cv=cv, scoring="accuracy")
score_pca = cross_val_score(pipe_pca, X, y, cv=cv, scoring="accuracy")

print(f"Sin PCA: media={score_no.mean():.4f}, desv?o={score_no.std():.4f}")
print(f"Con PCA ({n95} comp): media={score_pca.mean():.4f}, desv?o={score_pca.std():.4f}")


In [ ]:
n_vals = range(2, X.shape[1] + 1)
acc_vals = []
for n in n_vals:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("clf", LogisticRegression(max_iter=5000, multi_class="multinomial")),
    ])
    acc_vals.append(cross_val_score(pipe, X, y, cv=cv, scoring="accuracy").mean())

plt.figure(figsize=(7.5, 4.2))
plt.plot(list(n_vals), acc_vals, marker="o")
plt.xlabel("n_components")
plt.ylabel("Accuracy CV (media)")
plt.title("PCA + regresi?n log?stica")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
